# Assignment 2 - EDA and Data Structure

This notebook inspects the bus-level and zone-level load data for Assignment 2. It is designed for very large bus-level Parquet files, so it uses Parquet metadata, full zone-level reads, and bounded bus samples instead of loading all bus rows into memory.

Generated EDA tables and plots are saved in `solution/assignment_2/execution_outputs/`.

## Time-Series Leakage Policy

This project is strict time-series forecasting data. Do not randomly shuffle rows for sampling, validation, or train/test splits.

Policy for later modeling:

- Keep all 2025 target data out of model training.
- Use 2022-01-01 through 2024-12-31 for training/history features.
- Use 2025-01-01 through 2025-12-31 only as the test target period.
- For any forecast date, use only information available on or before `forecast_created_at`.
- EDA samples should be deterministic, time-ordered, or aggregate summaries. Do not random-shuffle time-series rows.

In [ ]:
from pathlib import Path
import os
import tempfile

os.environ.setdefault("MPLBACKEND", "Agg")
os.environ.setdefault("XDG_CACHE_HOME", str(Path(tempfile.gettempdir()) / "xdg-cache"))
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "matplotlib-cache"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

## Locate Assignment 2 Data

Expected files are yearly bus and zone Parquet files under `data/Assignment 2 - forecast/`.

In [ ]:
def find_repo_root(start: Path = Path.cwd()) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "README.md").exists() and (candidate / "data").exists():
            return candidate
    raise FileNotFoundError("Could not find repository root containing README.md and data/.")


repo_root = find_repo_root()
data_dir = repo_root / "data" / "Assignment 2 - forecast"
outputs_dir = repo_root / "solution" / "assignment_2" / "execution_outputs"
outputs_dir.mkdir(parents=True, exist_ok=True)

bus_files = sorted(data_dir.glob("bus_load_*.parquet"))
zone_files = sorted(data_dir.glob("zone_load_*.parquet"))
if not bus_files or not zone_files:
    raise FileNotFoundError(f"Expected bus_load_*.parquet and zone_load_*.parquet under {data_dir}")

file_inventory = pd.DataFrame(
    {
        "file": [str(p.relative_to(repo_root)) for p in [*bus_files, *zone_files]],
        "kind": ["bus"] * len(bus_files) + ["zone"] * len(zone_files),
        "size_mb": [round(p.stat().st_size / 1024 / 1024, 2) for p in [*bus_files, *zone_files]],
    }
)
file_inventory_path = outputs_dir / "assignment2_file_inventory.csv"
file_inventory.to_csv(file_inventory_path, index=False)
file_inventory

## Parquet Metadata and Scale

The bus files are too large for routine full-memory EDA. Use metadata for row counts and bounded samples for examples.

In [ ]:
metadata_rows = []
for path in [*bus_files, *zone_files]:
    pf = pq.ParquetFile(path)
    metadata_rows.append(
        {
            "file": path.name,
            "kind": "bus" if path.name.startswith("bus_") else "zone",
            "rows": pf.metadata.num_rows,
            "row_groups": pf.num_row_groups,
            "columns": ", ".join(pf.schema_arrow.names),
        }
    )

metadata_summary = pd.DataFrame(metadata_rows)
metadata_summary_path = outputs_dir / "assignment2_parquet_metadata_summary.csv"
metadata_summary.to_csv(metadata_summary_path, index=False)
metadata_summary

## Load Zone Data Fully and Bus Data as Deterministic Samples

Zone files are small enough to load fully. Bus data is sampled by taking the first Parquet batch from each yearly file, preserving natural file/time order and avoiding random shuffling.

In [ ]:
def read_first_batch(path: Path, columns=None, batch_size=200_000):
    pf = pq.ParquetFile(path)
    batches = pf.iter_batches(batch_size=batch_size, columns=columns)
    try:
        return next(batches).to_pandas()
    except StopIteration:
        return pd.DataFrame(columns=columns)


zone = pd.concat([pd.read_parquet(path) for path in zone_files], ignore_index=True)
bus_sample_columns = ["id", "bus_unique_id", "bus_type", "base_kv", "zone_name", "pd", "pg", "date", "he"]
bus_sample = pd.concat([read_first_batch(path, columns=bus_sample_columns) for path in bus_files], ignore_index=True)

print(f"Zone rows loaded: {len(zone):,}")
print(f"Bus sample rows loaded: {len(bus_sample):,}")
bus_sample.head()

## Standardize Columns and Timestamps

`HE` means hour-ending. In this assignment convention, `HE1` represents `00:00:00` to `00:59:59`, and `HE24` represents `23:00:00` to `23:59:59`. The notebook creates `interval_start_hour = he - 1` and an hourly timestamp at the interval start.

In [ ]:
def normalize_columns(df):
    return df.rename(columns={col: str(col).strip().lower() for col in df.columns})


def add_time_columns(df):
    out = normalize_columns(df).copy()
    out["date"] = pd.to_datetime(out["date"])
    out["he"] = pd.to_numeric(out["he"], errors="coerce").astype("Int64")
    out["interval_start_hour"] = out["he"] - 1
    out["timestamp"] = out["date"] + pd.to_timedelta(out["interval_start_hour"].astype(float), unit="h")
    out["year"] = out["date"].dt.year
    out["month"] = out["date"].dt.month
    out["day_of_week"] = out["date"].dt.dayofweek
    out["day_name"] = out["date"].dt.day_name()
    out["is_weekend"] = out["day_of_week"].isin([5, 6])
    return out


zone = add_time_columns(zone)
bus_sample = add_time_columns(bus_sample)

for numeric_col in ["pd", "pg", "load_bus_count", "gen_bus_count"]:
    if numeric_col in zone.columns:
        zone[numeric_col] = pd.to_numeric(zone[numeric_col], errors="coerce")
for numeric_col in ["pd", "pg", "base_kv"]:
    if numeric_col in bus_sample.columns:
        bus_sample[numeric_col] = pd.to_numeric(bus_sample[numeric_col], errors="coerce")

zone.head()

## Individual File EDA

Bus and zone files are profiled separately for each year because they represent different granularities. Zone files are small enough for full-file EDA; bus files are profiled with Parquet metadata plus a deterministic first-batch sample from each file.


In [ ]:
def parquet_date_range_from_stats(path: Path, column_name="date"):
    pf = pq.ParquetFile(path)
    names = pf.schema_arrow.names
    if column_name not in names:
        return pd.NaT, pd.NaT
    col_idx = names.index(column_name)
    mins, maxs = [], []
    for rg_idx in range(pf.num_row_groups):
        stats = pf.metadata.row_group(rg_idx).column(col_idx).statistics
        if stats is not None and stats.has_min_max:
            mins.append(stats.min)
            maxs.append(stats.max)
    if not mins:
        return pd.NaT, pd.NaT
    return pd.to_datetime(min(mins)), pd.to_datetime(max(maxs))


def compact_missing_summary(df, dataset_name, file_name, scope):
    return pd.DataFrame(
        {
            "dataset": dataset_name,
            "file": file_name,
            "scope": scope,
            "column": df.columns,
            "row_count": len(df),
            "missing_count": [int(df[col].isna().sum()) for col in df.columns],
            "missing_percent": [round(float(df[col].isna().mean() * 100), 3) for col in df.columns],
        }
    )


def example_rows(df, dataset_name, file_name, n=5):
    sample = df.head(n).copy()
    sample.insert(0, "example_row", range(1, len(sample) + 1))
    sample.insert(0, "file", file_name)
    sample.insert(0, "dataset", dataset_name)
    return sample

individual_profiles = []
individual_missingness = []
individual_examples = []

for path in zone_files:
    year = int(path.stem.split("_")[-1])
    raw = pd.read_parquet(path)
    df = add_time_columns(raw)
    for numeric_col in ["pd", "pg", "load_bus_count", "gen_bus_count"]:
        if numeric_col in df.columns:
            df[numeric_col] = pd.to_numeric(df[numeric_col], errors="coerce")
    individual_profiles.append(
        {
            "dataset": "zone_full_file",
            "kind": "zone",
            "year": year,
            "file": path.name,
            "scope": "full_file",
            "rows": len(df),
            "columns": df.shape[1],
            "row_groups": pq.ParquetFile(path).num_row_groups,
            "min_date": df["date"].min(),
            "max_date": df["date"].max(),
            "he_min": df["he"].min(),
            "he_max": df["he"].max(),
            "unique_timestamps": df["timestamp"].nunique(),
            "unique_zones": df["zone_name"].nunique() if "zone_name" in df.columns else np.nan,
            "unique_buses": np.nan,
            "total_pd": df["pd"].sum() if "pd" in df.columns else np.nan,
            "mean_pd": df["pd"].mean() if "pd" in df.columns else np.nan,
        }
    )
    individual_missingness.append(compact_missing_summary(df, "zone_full_file", path.name, "full_file"))
    individual_examples.append(example_rows(df, "zone_full_file", path.name))

for path in bus_files:
    year = int(path.stem.split("_")[-1])
    pf = pq.ParquetFile(path)
    sample = read_first_batch(path, columns=bus_sample_columns, batch_size=200_000)
    sample = add_time_columns(sample)
    for numeric_col in ["pd", "pg", "base_kv"]:
        if numeric_col in sample.columns:
            sample[numeric_col] = pd.to_numeric(sample[numeric_col], errors="coerce")
    min_date, max_date = parquet_date_range_from_stats(path)
    individual_profiles.append(
        {
            "dataset": "bus_sample_file",
            "kind": "bus",
            "year": year,
            "file": path.name,
            "scope": "parquet_metadata_plus_first_200k_rows",
            "rows": pf.metadata.num_rows,
            "columns": len(pf.schema_arrow.names),
            "row_groups": pf.num_row_groups,
            "min_date": min_date if pd.notna(min_date) else sample["date"].min(),
            "max_date": max_date if pd.notna(max_date) else sample["date"].max(),
            "he_min": sample["he"].min(),
            "he_max": sample["he"].max(),
            "unique_timestamps": sample["timestamp"].nunique(),
            "unique_zones": sample["zone_name"].nunique() if "zone_name" in sample.columns else np.nan,
            "unique_buses": sample["bus_unique_id"].nunique() if "bus_unique_id" in sample.columns else np.nan,
            "total_pd": sample["pd"].sum() if "pd" in sample.columns else np.nan,
            "mean_pd": sample["pd"].mean() if "pd" in sample.columns else np.nan,
        }
    )
    individual_missingness.append(compact_missing_summary(sample, "bus_sample_file", path.name, "first_200k_rows"))
    individual_examples.append(example_rows(sample, "bus_sample_file", path.name))

individual_file_profiles = pd.DataFrame(individual_profiles).sort_values(["kind", "year"])
individual_file_missingness = pd.concat(individual_missingness, ignore_index=True)
individual_file_examples = pd.concat(individual_examples, ignore_index=True, sort=False)

individual_file_profiles_path = outputs_dir / "assignment2_individual_file_profiles.csv"
individual_file_missingness_path = outputs_dir / "assignment2_individual_file_missingness.csv"
individual_file_examples_path = outputs_dir / "assignment2_individual_file_examples.csv"

individual_file_profiles.to_csv(individual_file_profiles_path, index=False)
individual_file_missingness.to_csv(individual_file_missingness_path, index=False)
individual_file_examples.to_csv(individual_file_examples_path, index=False)

individual_file_profiles


### Per-File Missingness

The table below keeps zone full-file missingness and bus deterministic-sample missingness separate. Bus missingness is not interpreted as a full-file count unless it comes from Parquet metadata.


In [ ]:
individual_file_missingness.sort_values(
    ["missing_percent", "kind" if "kind" in individual_file_missingness.columns else "dataset", "file", "column"],
    ascending=[False, True, True, True],
).head(40)


### Per-File Example Values

These examples are for structure inspection only. They are not random samples and should not be used as modeling splits.


In [ ]:
individual_file_examples.head(40)


## Data Structure: Granularity, Counts, and Coverage

## Zone-to-Bus Reconciliation Check

Zone and bus files are different granularities. As a bounded integrity check, this section reads one full day of bus-level records and compares summed bus `pd` against zone-level `pd` for the same zone, date, and HE.

In [ ]:
reconcile_date = "2022-01-01"
bus_reconcile = pd.read_parquet(
    bus_files[0],
    columns=["bus_unique_id", "zone_name", "pd", "date", "he"],
    filters=[("date", "==", reconcile_date)],
)
zone_reconcile = pd.read_parquet(
    zone_files[0],
    columns=["zone_name", "pd", "date", "he"],
    filters=[("date", "==", reconcile_date)],
)

bus_reconcile["date"] = pd.to_datetime(bus_reconcile["date"])
zone_reconcile["date"] = pd.to_datetime(zone_reconcile["date"])
bus_sum = bus_reconcile.groupby(["zone_name", "date", "he"], as_index=False)["pd"].sum().rename(columns={"pd": "bus_sum_pd"})
zone_day = zone_reconcile.rename(columns={"pd": "zone_pd"})
zone_bus_reconciliation = zone_day.merge(bus_sum, on=["zone_name", "date", "he"], how="left")
zone_bus_reconciliation["abs_diff_pd"] = (zone_bus_reconciliation["zone_pd"] - zone_bus_reconciliation["bus_sum_pd"]).abs()
zone_bus_reconciliation["pct_diff"] = zone_bus_reconciliation["abs_diff_pd"] / zone_bus_reconciliation["zone_pd"].abs().replace(0, np.nan)
zone_bus_reconciliation_path = outputs_dir / "assignment2_zone_bus_reconciliation_sample.csv"
zone_bus_reconciliation.to_csv(zone_bus_reconciliation_path, index=False)
zone_bus_reconciliation.sort_values("abs_diff_pd", ascending=False).head(20)

In [ ]:
structure_summary = pd.DataFrame(
    [
        {
            "dataset": "bus_metadata",
            "rows": int(metadata_summary.loc[metadata_summary["kind"] == "bus", "rows"].sum()),
            "columns": len(pq.ParquetFile(bus_files[0]).schema_arrow.names),
            "min_date": bus_sample["date"].min(),
            "max_date": bus_sample["date"].max(),
            "unique_timestamps_in_sample": bus_sample["timestamp"].nunique(),
            "unique_buses_in_sample": bus_sample["bus_unique_id"].nunique(),
            "unique_zones_in_sample": bus_sample["zone_name"].nunique(),
        },
        {
            "dataset": "zone_full",
            "rows": len(zone),
            "columns": zone.shape[1],
            "min_date": zone["date"].min(),
            "max_date": zone["date"].max(),
            "unique_timestamps": zone["timestamp"].nunique(),
            "unique_zones": zone["zone_name"].nunique(),
        },
    ]
)
structure_summary_path = outputs_dir / "assignment2_structure_summary.csv"
structure_summary.to_csv(structure_summary_path, index=False)
structure_summary

In [ ]:
zone_counts_by_date = zone.groupby("date")["zone_name"].nunique().reset_index(name="zone_count")
zone_bus_counts = (
    zone.groupby("zone_name", as_index=False)
    .agg(
        avg_load_bus_count=("load_bus_count", "mean"),
        max_load_bus_count=("load_bus_count", "max"),
        avg_gen_bus_count=("gen_bus_count", "mean"),
        max_gen_bus_count=("gen_bus_count", "max"),
    )
    .sort_values("avg_load_bus_count", ascending=False)
)
bus_sample_counts = bus_sample.groupby("zone_name")["bus_unique_id"].nunique().reset_index(name="sample_unique_bus_count")

zone_counts_by_date.to_csv(outputs_dir / "assignment2_zone_counts_by_date.csv", index=False)
zone_bus_counts.to_csv(outputs_dir / "assignment2_zone_bus_count_summary.csv", index=False)
bus_sample_counts.to_csv(outputs_dir / "assignment2_bus_sample_counts_by_zone.csv", index=False)

display(zone_bus_counts.head(20))
display(bus_sample_counts.head(20))

## Missingness

Zone missingness is computed on the full zone dataset. Bus missingness is computed on the deterministic bus sample because full bus data is too large for notebook-scale EDA.

In [ ]:
def missing_summary(df, dataset_name):
    return pd.DataFrame(
        {
            "dataset": dataset_name,
            "column": df.columns,
            "row_count": len(df),
            "missing_count": [df[col].isna().sum() for col in df.columns],
            "missing_percent": [round(df[col].isna().mean() * 100, 3) for col in df.columns],
        }
    )


missingness = pd.concat(
    [missing_summary(zone, "zone_full"), missing_summary(bus_sample, "bus_sample")],
    ignore_index=True,
)
missingness_path = outputs_dir / "assignment2_missingness_summary.csv"
missingness.to_csv(missingness_path, index=False)
missingness.sort_values(["missing_percent", "dataset", "column"], ascending=[False, True, True]).head(30)

## Zone-Level Aggregation and Bus-Level Granularity

Zone data is already aggregated and includes `load_bus_count`. For bus-level data, this notebook uses bounded samples for structural inspection. Exact full bus aggregation should be done in later pipeline code with chunked/Parquet-aware processing.

In [ ]:
zone_hourly_granularity = (
    zone.groupby(["date", "he"])
    .agg(zone_count=("zone_name", "nunique"), total_pd=("pd", "sum"), total_load_bus_count=("load_bus_count", "sum"))
    .reset_index()
)
zone_hourly_granularity_path = outputs_dir / "assignment2_zone_hourly_granularity.csv"
zone_hourly_granularity.to_csv(zone_hourly_granularity_path, index=False)
zone_hourly_granularity.head()

## Plot Helpers

In [ ]:
def save_current_plot(filename):
    path = outputs_dir / filename
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches="tight")
    print(f"Saved plot to {path}")
    plt.close()

## Plot: Hourly Load Curves

In [ ]:
hourly_zone_load = zone.groupby("he", as_index=False)["pd"].mean().sort_values("he")
plt.figure(figsize=(10, 5))
plt.plot(hourly_zone_load["he"], hourly_zone_load["pd"], marker="o")
plt.title("Average Zone Load by Hour Ending")
plt.xlabel("Hour Ending (HE)")
plt.ylabel("Average PD (MW)")
plt.xticks(range(1, 25))
plt.grid(True, alpha=0.3)
save_current_plot("assignment2_hourly_zone_load_curve.png")

if bus_sample["pd"].notna().any():
    hourly_bus_sample_load = bus_sample.groupby("he", as_index=False)["pd"].mean().sort_values("he")
    plt.figure(figsize=(10, 5))
    plt.plot(hourly_bus_sample_load["he"], hourly_bus_sample_load["pd"], marker="o", color="tab:green")
    plt.title("Sampled Bus Load by Hour Ending")
    plt.xlabel("Hour Ending (HE)")
    plt.ylabel("Average PD (MW, sample)")
    plt.xticks(range(1, 25))
    plt.grid(True, alpha=0.3)
    save_current_plot("assignment2_hourly_bus_sample_load_curve.png")

## Plot: Weekday vs Weekend Behavior

In [ ]:
weekday_weekend = zone.groupby(["is_weekend", "he"], as_index=False)["pd"].mean().sort_values(["is_weekend", "he"])
plt.figure(figsize=(10, 5))
for is_weekend, group in weekday_weekend.groupby("is_weekend"):
    label = "Weekend" if is_weekend else "Weekday"
    plt.plot(group["he"], group["pd"], marker="o", label=label)
plt.title("Average Zone Load: Weekday vs Weekend")
plt.xlabel("Hour Ending (HE)")
plt.ylabel("Average PD (MW)")
plt.xticks(range(1, 25))
plt.legend()
plt.grid(True, alpha=0.3)
save_current_plot("assignment2_weekday_weekend_zone_load.png")

## Plot: Annual Seasonality

In [ ]:
daily_zone_load = zone.groupby("date", as_index=False)["pd"].sum().sort_values("date")
daily_zone_load.to_csv(outputs_dir / "assignment2_daily_total_zone_load.csv", index=False)

plt.figure(figsize=(12, 5))
plt.plot(daily_zone_load["date"], daily_zone_load["pd"], linewidth=1)
plt.title("Daily Total Zone Load Over Time")
plt.xlabel("Date")
plt.ylabel("Total PD (MW)")
plt.grid(True, alpha=0.3)
save_current_plot("assignment2_annual_seasonality_daily_zone_load.png")

monthly_zone_load = zone.groupby(["year", "month"], as_index=False)["pd"].mean()
monthly_pivot = monthly_zone_load.pivot(index="month", columns="year", values="pd")
monthly_pivot.to_csv(outputs_dir / "assignment2_monthly_seasonality_table.csv")
monthly_pivot

## Plot: Zone Distributions

In [ ]:
zone_distribution = zone.groupby("zone_name", as_index=False)["pd"].mean().sort_values("pd", ascending=False)
zone_distribution.to_csv(outputs_dir / "assignment2_zone_average_pd_distribution.csv", index=False)

plt.figure(figsize=(12, 6))
plt.bar(zone_distribution["zone_name"].astype(str), zone_distribution["pd"])
plt.title("Average Load by Zone")
plt.xlabel("Zone")
plt.ylabel("Average PD (MW)")
plt.xticks(rotation=45, ha="right")
plt.grid(axis="y", alpha=0.3)
save_current_plot("assignment2_zone_average_pd_distribution.png")

## Time-Based Split Markers for Later Modeling

This cell does not create modeling splits yet. It records the policy boundaries to prevent accidental leakage in future notebooks.

In [ ]:
split_policy = pd.DataFrame(
    [
        {
            "purpose": "training_history",
            "start_date": "2022-01-01",
            "end_date": "2024-12-31",
            "rule": "May be used for model fitting and historical feature creation.",
        },
        {
            "purpose": "test_target_holdout",
            "start_date": "2025-01-01",
            "end_date": "2025-12-31",
            "rule": "Do not use for training; reserve for next-day and next-month evaluation targets.",
        },
    ]
)
split_policy_path = outputs_dir / "assignment2_time_series_split_policy.csv"
split_policy.to_csv(split_policy_path, index=False)
split_policy